In [1]:
import pandas as pd
import numpy as np
import pickle
import ast
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load model

In [2]:
# Load Vietnamese SBERT model
model = SentenceTransformer('keepitreal/vietnamese-sbert')
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

   Embedding dimension: 768


### Load embeddings

In [3]:
with open("recipes_embeddings_list.pkl", "rb") as f:
    recipes_embeddings_list = pickle.load(f)
print(f"Loaded {len(recipes_embeddings_list)} recipe embeddings")

Loaded 10263 recipe embeddings


### Load dataset

In [4]:
all_recipes_df = pd.read_csv("../../data/all_recipes_final.csv")

print(f"Loaded dataset: {len(all_recipes_df)} recipes")
print(f"Columns: {all_recipes_df.columns.tolist()}")

Loaded dataset: 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


### 3. Late Fusion Search Function

**Late Fusion Strategy:**
1. Tính similarity của query với **TẤT CẢ** các câu trong mỗi món
2. Lấy **trung bình** (average) similarity của tất cả câu → điểm số của món
3. Rank tất cả món theo điểm trung bình → Top K

In [5]:
def search_recipes_late_fusion(query, model, recipes_embeddings_list, all_recipes_df, top_k=10):
    """
    Search recipes using LATE FUSION strategy (Average Similarity)

    Late Fusion = Tính similarity với TẤT CẢ câu trong món → Average → Rank

    Args:
        query: User's search query (Vietnamese)
        model: SentenceTransformer model
        recipes_embeddings_list: List of embeddings per dish
        all_recipes_df: Recipe metadata dataframe
        top_k: Number of results to return

    Returns:
        DataFrame with top_k recipes and similarity scores
    """
    # 1. Encode query
    query_embedding = model.encode([query])
    query_embedding = query_embedding / np.linalg.norm(query_embedding)  # Normalize

    # 2. Calculate average similarity for EACH recipe
    recipe_scores = []

    for recipe_idx, dish_embeds in enumerate(recipes_embeddings_list):
        if len(dish_embeds) == 0:
            continue

        # Normalize dish embeddings
        dish_embeds_norm = dish_embeds / np.linalg.norm(dish_embeds, axis=1, keepdims=True)

        # Compute cosine similarity với TẤT CẢ câu
        similarities = np.dot(dish_embeds_norm, query_embedding.T).flatten()

        # LATE FUSION: Average similarity
        avg_similarity = np.mean(similarities)

        recipe_scores.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': float(avg_similarity),
            'max_similarity': float(np.max(similarities)),
            'min_similarity': float(np.min(similarities)),
            'num_sentences': len(similarities)
        })

    # 3. Sort by average similarity
    recipe_scores.sort(key=lambda x: x['avg_similarity'], reverse=True)
    top_recipes = recipe_scores[:top_k]

    # 4. Create results dataframe with FULL recipe info
    results = []
    for item in top_recipes:
        recipe_idx = item['recipe_idx']
        recipe = all_recipes_df.iloc[recipe_idx]

        results.append({
            'recipe_idx': recipe_idx,
            'avg_similarity': item['avg_similarity'],
            'max_similarity': item['max_similarity'],
            'min_similarity': item['min_similarity'],
            'num_sentences': item['num_sentences'],
            'title': recipe['title'],
            'type_of_food': recipe['type_of_food'],
            'cook_time': recipe['cook_time'],
            'num_of_people': recipe['num_of_people'],
            'ingredients': recipe['ingredients'],
            'step': recipe['step'],
            'note': recipe['note'],
            'description': recipe['description'],
            'link': recipe['link']  # Thêm link
        })

    return pd.DataFrame(results)

### 4. Display results function

In [6]:
def parse_list_field(field_value):
    """
    Parse string / list field to Python list safely.
    """
    if pd.isna(field_value):
        return []

    if isinstance(field_value, list):
        return field_value

    if isinstance(field_value, str):
        try:
            parsed = ast.literal_eval(field_value)
            if isinstance(parsed, list):
                return parsed
            return []
        except (ValueError, SyntaxError):
            # fallback: split by comma
            return [s.strip() for s in field_value.split(",") if s.strip()]

    return []

In [7]:
import re

def display_results(results_df, query):
    """
    Display search results với TẤT CẢ thông tin món ăn

    Args:
        results_df: DataFrame from search_recipes_late_fusion
        query: Original query string
    """
    print(f"Query: '{query}'")
    print(f"Top {len(results_df)} Results:")

    for idx, row in results_df.iterrows():
        print(f"\n{'='*100}")
        print(f"{idx+1}. [{row['avg_similarity']:.4f}] {row['title']}")
        print(f"{'='*100}")

        # Basic info
        print(f"\nTHÔNG TIN CƠ BẢN:")
        print(f"   • Loại món: {row['type_of_food']}")
        print(f"   • Thời gian nấu: {row['cook_time']}")
        print(f"   • Số người ăn: {row['num_of_people']}")
        
        # Link
        if pd.notna(row['link']):
            print(f"   • Link: {row['link']}")

        # Similarity scores
        print(f"\nĐIỂM SIMILARITY:")
        print(f"   • Trung bình (AVG): {row['avg_similarity']:.4f}")
        print(f"   • Cao nhất (MAX): {row['max_similarity']:.4f}")
        print(f"   • Thấp nhất (MIN): {row['min_similarity']:.4f}")
        print(f"   • Số câu đánh giá: {row['num_sentences']}")

        # Description
        if pd.notna(row['description']):
            print(f"\nMÔ TẢ:")
            print(f"   {row['description']}")

        # Ingredients
        ingredients = parse_list_field(row['ingredients'])
        if ingredients:
            print(f"\nNGUYÊN LIỆU ({len(ingredients)} món):")
            for i, ing in enumerate(ingredients, 1):
                print(f"   {i}. {ing}")

        # Steps
        steps = parse_list_field(row['step'])
        if steps:
            # 1. Gộp tất cả step thành 1 chuỗi
            steps_text = " ".join(step.strip() for step in steps)

            # 2. Format: gặp "Bước X:" thì xuống dòng
            steps_text = re.sub(r'(Bước\s+\d+:)', r'\n\1', steps_text).strip()

            print(f"\nCÁCH LÀM:")
            print(steps_text)

        # Notes
        notes = parse_list_field(row['note'])
        if notes:
            print(f"\nLƯU Ý:")
            for i, note in enumerate(notes, 1):
                print(f"   • {note}")

        print()

In [8]:
# Test Late Fusion
test_queries = [
    "Món ăn có thịt bò nấu nhanh",
]

# Run Late Fusion tests
for query in test_queries:
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=5
    )

    # Display results
    display_results(results, query)

Query: 'Món ăn có thịt bò nấu nhanh'
Top 5 Results:

1. [0.6043] Bò hầm cà rốt

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 30phút
   • Số người ăn: 2
   • Link: https://vncooking.com/cong-thuc/bo-ham-ca-rot-14

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6043
   • Cao nhất (MAX): 0.7382
   • Thấp nhất (MIN): 0.4551
   • Số câu đánh giá: 4

MÔ TẢ:
   Bò luôn là món thịt mà đa phần các gia đình Việt rất ưa chuộng, bò chứa hàm lượng dinh dưỡng siêu cao cộng với cà rốt nữa làm tăng thêm phần dinh dưỡng của món ăn. Cùng chuẩn bị nguyên liệu thực hiện món Bò hầm cà rốt này nhé.

NGUYÊN LIỆU (7 món):
   1. Thịt bò 150 gram
   2. Cà rốt 2 củ
   3. Nước mắm 1 muỗng cafe
   4. Gừng 1 củ
   5. Muối 1 muỗng
   6. Dầu ăn 2 muỗng
   7. Sả 1 cây

CÁCH LÀM:
Bước 1: Nguyên liệu rửa sạch. Cà rốt cạo vỏ thái thành những hình vuông nhỏ, vừa ăn. Thịt bò cũng vậy, thái thành từng miếng hình vuống nhỏ thôi cho không bị day nha. Các nguyên liệu khác bỏ vỏ đun trên 1 nồi nước nhỏ, chờ khi nướ

### 5. Interactive Search

In [9]:
# Interactive search với Late Fusion
print("Enter your query (type 'quit' to exit):\n")

while True:
    query = input("Query: ").strip()

    if query.lower() in ['quit', 'exit', 'q']:
        break

    if not query:
        continue

    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=10
    )

    # Display results
    display_results(results, query)

Enter your query (type 'quit' to exit):

Query: 'gà chiên'
Top 10 Results:

1. [0.5814] Thịt gà chiên giòn trên từng miếng thịt

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 45phút
   • Số người ăn: 4
   • Link: https://vncooking.com/cong-thuc/thit-ga-chien-gion-tren-tung-mieng-thit-60

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.5814
   • Cao nhất (MAX): 0.6269
   • Thấp nhất (MIN): 0.5390
   • Số câu đánh giá: 4

MÔ TẢ:
   Thịt gà là món ăn được yêu thích tại Việt Nam, chế biến món thịt gà bằng công thức mà VN Cooking cung cấp cho bạn sẽ đảm bảo được vị ngon nhất để bạn có thể chiêu đãi bạn bè, gia đình bữa ăn tuyệt hảo nhất.

NGUYÊN LIỆU (6 món):
   1. Ức gà 300 gram
   2. Bột chiên giòn 50 gram
   3. Bột chiên xù 50 gram
   4. Trứng gà 2 quả
   5. Gừng 1 củ
   6. Rượu vang 1 muỗng

CÁCH LÀM:
Bước 1: - Thịt gà  mua về rửa sạch, loại bỏ phần da và mỡ. - Thái thành những miếng dài. - Cho 1 muỗng nước tương, 1 muỗng rượu vang, 1 muỗng muối, 1/2 muỗng tiêu và gừng thái s

In [10]:
results.head()

,recipe_idx,avg_similarity,max_similarity,min_similarity,num_sentences,title,type_of_food,cook_time,num_of_people,ingredients,step,note,description,link
0,1225,0.581386,0.626933,0.539004,4,Thịt gà chiên giòn trên từng miếng thịt,Món chính,45phút,4,"['Ức gà 300 gram', 'Bột chiên giòn 50 gram', '...","['Bước 1: - Thịt gà mua về rửa sạch, loại bỏ ...",[],"Thịt gà là món ăn được yêu thích tại Việt Nam,...",https://vncooking.com/cong-thuc/thit-ga-chien-...
1,6775,0.572879,0.658104,0.516972,5,Gà tẩm bột chiên giòn Aji-Quick giòn rụm bằng ...,Món chiên,60 phút,4 người,"['1 kg Thịt gà (xương gà hoặc các phần khác)',...","['Bước 1: Sơ chế thịt gà: Thịt gà mua về, bạn ...",['Xem chi tiết: Cách chọn mua gà ngon và cách ...,Bạn đang tìm cách làm gà tẩm bột chiên giòn Aj...,https://www.dienmayxanh.com/vao-bep/cach-lam-g...
2,1170,0.554375,0.620303,0.469262,4,Gà chiên sốt cay da giòn,Món chính,45phút,4,"['Gà phi lê 500 gram', 'Bột bắp 200 gram', 'Tr...",['Bước 1: - Gà thái miếng ướp với 1 muỗng muối...,[],Những miếng gà chiên giòn được tưới thêm nước ...,https://vncooking.com/cong-thuc/ga-chien-sot-c...
3,4689,0.552033,0.677593,0.462724,8,Gà nướng mật ong nồi chiên không dầu tiện lợi ...,Món nướng,45 phút,1 con gà (khoảng 1.2kg),"['1 con Gà (khoảng 1.2 kg)', '1 ít Gia vị ướp ...",['Bước 1: Sơ chế và chuẩn bị gà nguyên con: Gà...,['Xem chi tiết: Cách chọn mua gà ngon và cách ...,Bạn đang tìm kiếm công thức gà nướng mật ong n...,https://www.dienmayxanh.com/vao-bep/cach-lam-g...
4,1072,0.549949,0.620867,0.468054,4,Gà bó xôi chiên giòn,Món chính,60phút,3,"['Gạo nếp 300 gram', 'Trứng gà 1 quả', 'Thịt g...",['Bước 1: - Gạo ngâm qua đêm rồi đem nấu cùng ...,[],"Gà ướp với gia vị, được bao quanh là xôi dẻo t...",https://vncooking.com/cong-thuc/ga-bo-xoi-chie...


### 6. Combine with state.json

#### 6.1 Load state.json

In [21]:
from pathlib import Path
import json

STATE_FILE = "state.json"

def load_state_json(filepath=STATE_FILE):
    """Load dialogue state from JSON file"""
    try:
        if Path(filepath).exists():
            with open(filepath, 'r', encoding='utf-8') as f:
                state = json.load(f)
            return state
        else:
            print(f"[Error]: {filepath} not found")
            return None
    except json.JSONDecodeError:
        print(f"[Error]: Invalid JSON in {filepath}")
        return None

In [30]:
state = load_state_json()
state

{'hard_constraints': {'type_of_food': ['món kho'],
  'ingredients': ['thịt heo']},
 'soft_constraints': {'cook_time': ['45'],
  'num_of_people': ['4'],
  'calories': ['none'],
  'algeric': ['none']},
 'recommended_items': [],
 'accepted_items': [],
 'rejected_items': []}

#### 6.2 Generate query from state (Rule-based)

In [31]:
def generate_query_from_state(state):
    """
    Generate search query from state.json constraints using rule-based approach
    
    Args:
        state: Dictionary from state.json (with hard_constraints, soft_constraints)
    
    Returns:
        Generated query string
    """
    parts = []
    
    # Hard constraints (priority)
    if "hard_constraints" in state:
        # Type of food
        if state["hard_constraints"].get("type_of_food"):
            type_food = state["hard_constraints"]["type_of_food"]
            if type_food and type_food[0] != "none":
                parts.append(type_food[0])
        
        # Ingredients
        if state["hard_constraints"].get("ingredients"):
            ingredients = state["hard_constraints"]["ingredients"]
            if ingredients and ingredients != ["none"]:
                if len(ingredients) == 1:
                    parts.append(f"có {ingredients[0]}")
                else:
                    parts.append(f"có {', '.join(ingredients)}")
    
    # Soft constraints
    if "soft_constraints" in state:
        # Number of people
        if state["soft_constraints"].get("num_of_people"):
            num_people = state["soft_constraints"]["num_of_people"]
            if num_people and num_people[0] != "none":
                parts.append(f"cho {num_people[0]} người")
        
        # Cook time
        if state["soft_constraints"].get("cook_time"):
            cook_time = state["soft_constraints"]["cook_time"]
            if cook_time and cook_time[0] != "none":
                parts.append(f",có thời gian nấu {cook_time[0]} phút")
    
    # Combine parts into natural query
    if not parts:
        return "Món ăn"
    
    query = " ".join(parts)
    return query

In [32]:
# Test query generation
test_query = generate_query_from_state(state)
print(f"Generated query: '{test_query}'")

Generated query: 'món kho có thịt heo cho 4 người ,có thời gian nấu 45 phút'


#### 6.3 Search from state.json

In [38]:
def search_from_state_json(state_filepath=STATE_FILE, top_k=10):
    """
    Args:
        state_filepath: Path to state.json file
        top_k: Number of results to return
    
    Returns:
        DataFrame with search results
    """
    # STEP 1: Load state.json
    print("\nSTEP 1: Loading state.json...")
    print("-"*100)
    state = load_state_json(state_filepath)
    
    if state is None:
        return None
    
    # STEP 2: Generate query
    print("\nSTEP 2: Generating query from constraints...")
    
    
    query = generate_query_from_state(state)
    print(f"Generated Query: '{query}'")
    print("-"*100)
    
    # STEP 3: Search recipes using Late Fusion
    print("\nSTEP 3: Searching recipes with Late Fusion...")
    
    results = search_recipes_late_fusion(
        query=query,
        model=model,
        recipes_embeddings_list=recipes_embeddings_list,
        all_recipes_df=all_recipes_df,
        top_k=top_k
    )
    
    print(f"Found {len(results)} recipes")
    print("-"*100)

    # STEP 4: Display results
    print("STEP 4: RESULTS")
    display_results(results, query)
    
    return results

In [39]:
# Run search from state.json
results = search_from_state_json(
    state_filepath="state.json",
    top_k=10
)


STEP 1: Loading state.json...
----------------------------------------------------------------------------------------------------

STEP 2: Generating query from constraints...
Generated Query: 'món kho có thịt heo cho 4 người ,có thời gian nấu 45 phút'
----------------------------------------------------------------------------------------------------

STEP 3: Searching recipes with Late Fusion...
Found 10 recipes
----------------------------------------------------------------------------------------------------
STEP 4: RESULTS
Query: 'món kho có thịt heo cho 4 người ,có thời gian nấu 45 phút'
Top 10 Results:

1. [0.6104] Thịt heo kho cùi dừa

THÔNG TIN CƠ BẢN:
   • Loại món: Món chính
   • Thời gian nấu: 45phút
   • Số người ăn: 4
   • Link: https://vncooking.com/cong-thuc/thit-heo-kho-cui-dua-247

ĐIỂM SIMILARITY:
   • Trung bình (AVG): 0.6104
   • Cao nhất (MAX): 0.8567
   • Thấp nhất (MIN): 0.4705
   • Số câu đánh giá: 4

MÔ TẢ:
   Món thịt kho tàu hay thịt kho hột vịt quá quen 